# 3D Fluorescence Nuclei Segmentation


> **Python 3.10, 3.11, 3.12, 3.13, or 3.14.** This notebook and `requirements.txt` support all five; the pins that differ by version are handled automatically via markers in `requirements.txt` (see its header comment and README.md's Prerequisites section). Cell 1 below prints which Python version it detected and whether it's a supported one. Install with `uv pip install -r requirements.txt --override overrides.txt` (`uv` required -- see README.md for why).
>
> **Python 3.14 limitations:** TensorFlow has no Python 3.14 wheels, so TensorFlow, `csbdeep` and `stardist` are not installed there and **StarDist is unavailable**. If `trig_stardist = True` is set on 3.14, Cell 3 prints a warning and automatically sets it back to `False`. Likewise, `aicspylibczi` (the Zeiss reader) has no 3.14 wheels, so **`.czi` files can't be opened on 3.14** (Cell 4 raises a clear error) — use Python 3.10–3.13 for `.czi`, or export to OME-TIFF. Watershed, Cellpose 3D, Cellpose for cell shape and all other file formats work normally.

---


This notebook is a **generic template** for 3D fluorescence image segmentation and quantification of nuclei and cell-associated markers.

**Before running the workflow:**
1. Set the file path, ROI and experiment name in **Cell 3**.
2. Run Cell 4, fill in the stain table (channel, color and marker for each condition) and click **Confirm**.
3. Start with the default processing settings and tune only the parameters that need adjusting for your dataset.

**Segmentation methods available:**
- **Watershed** (default) — classical multi-scale erosion + EDT peaks. Splitting of touching nuclei is automatic (built-in first pass, then size-based iterative re-splitting — see Cell 3, section 6).
- **Cellpose 3D** — deep-learning model; handles touching nuclei natively. Set `trig_cellpose = True`.
- **StarDist** — 2D-to-3D translation model. Set `trig_stardist = True`. *Not available on Python 3.14 (no TensorFlow wheels) — automatically disabled with a warning.*
- **Cellpose for cell shape** — independent of the nuclei method above, set `trig_cellpose_cyto = True` to shape the cytoplasm/cell body with Cellpose 3D instead of the nuclei-seeded watershed (requires a `CYTOPLASM` channel). For example, combine `trig_stardist = True` with `trig_cellpose_cyto = True` to find nuclei with StarDist and shape cells with Cellpose (on Python 3.14, use `trig_cellpose = True` for the nuclei instead).

---

# 1. Import Libraries and Helpers

Load the Python packages and the helper functions.

**Cell 1** (Import libraries)

Loads all required packages (napari, scipy, scikit-image, ...) and prints the detected Python version.

In [ ]:
from helpers.notebook_setup_helpers import load_nuclei_notebook_setup
globals().update(load_nuclei_notebook_setup())

**Cell 2** (Load helper functions)

Loads the functions from `helpers/notebook_helpers.py`. Re-run it after editing that file (no kernel restart needed).

In [ ]:
from helpers.notebook_helpers import reload_helpers
reload_helpers()

# 2. Inputs and Setup

Your settings and the stain table. For a new dataset, these are the only cells you need to edit.

**Cell 3** (Settings)

Input file (or a folder of images for **batch mode**), ROI, experiment name, object sizes, segmentation method and nuclei size refinement. Each setting is explained next to it.

- **Segmentation method:** leave `trig_cellpose` and `trig_stardist` both `False` for watershed. StarDist is not available on Python 3.14 (it is switched off with a warning).
- **Size refinement:** after segmentation, labels much bigger than a typical nucleus are split again with increasingly aggressive settings, and small fragments are merged into a touching neighbour when they look like the same nucleus. Labels are never deleted.

In [ ]:
# --- 1. Input -----------------------------------------------------------------
# Image file (.nd2, .tif/.ome.tif, .czi, .lif, .oib) or a FOLDER of images
# (batch mode: outputs go to '<folder>_output/<image name>/').
input_file = 'path/to/your_image_file.nd2'

# ROI = [x0, x1, y0, y1, z0, z1]; 0 = keep the full range.
ROI = [0, 0, 0, 0, 0, 0]
# True = drag the ROI (X/Y) in a napari window before loading.
interactive_roi = False

# --- 2. Experiment -------------------------------------------------------------
name_setup = 'your_experiment_name'  # prefix of the saved setup/output files
use_setup = True            # reuse the saved stain table and contrast/gamma
automatic_contrast = False  # True = pick contrast automatically (no napari window)

# --- 3. Sizes -----------------------------------------------------------------
nuclei_diameter = 10  # um
cell_diameter = 30    # um
scale_factor = 1.0
zoom_factors = [1.0, 1.0, 1.0]  # [Z, Y, X]

# --- 4. Segmentation method ---------------------------------------------------
# Both False = watershed (default).
trig_cellpose = False  # Cellpose 3D for nuclei (GPU recommended)
trig_stardist = False  # StarDist for nuclei (not available on Python 3.14)
trig_stardist = resolve_stardist_flag(trig_stardist)
trig_cellpose_cyto = False  # Cellpose 3D for the cell shape (needs a CYTOPLASM channel)

# --- 5. Quantification ----------------------------------------------------------
multilabel = True            # multi-marker combinations in the summary
aggregate_grow_factor = 2.0  # growth used to detect cell aggregates

# --- 6. Nuclei size refinement (automatic splitting/merging) ---------------------
size_reference = "blend"  # reference nucleus size: "blend", "input" or "detected"
iterative_split = True    # re-split labels > oversize_factor x reference
oversize_factor = 2.0
max_split_iterations = 3  # 1-3 escalating split levels
merge_undersized = True   # merge fragments < undersize_factor x reference
undersize_factor = 0.5

**Cell 4** (Load image and stain table)

1. Loads the image and reads voxel size and metadata (large files are read in chunks automatically). With `interactive_roi = True`, a napari window opens first to drag the ROI. In batch mode nothing is loaded: the channels of all images are checked instead, and any image with a different channel list is flagged.
2. Prints the channel list.
3. Shows the **stain table**. For each condition choose the channel (`no` = not used) and color, type the marker name, and tick **Intracellular?** for markers that define the cytoplasm when there is no CYTOPLASM channel. Click **Confirm**: the table is saved and reused next time (`use_setup = True`).

In [ ]:
batch_mode = is_batch_input(input_file)
file_meta = None

if batch_mode:
    print(f"Batch mode: {len(list_image_files(input_file))} image(s) will be "
          f"processed by Cell 5.")
else:
    if interactive_roi:
        ROI = select_roi_interactively(input_file, ROI, napari_module=napari)
    (meta, img, r_X, r_Y, r_Z, file_meta, ROI_print,
     cyto_factor, PCM_factor, zoom_factors, used_lazy_loading) = initialize_dataset(
        input_file, ROI,
        nuclei_diameter=nuclei_diameter,
        cell_diameter=cell_diameter,
        scale_factor=scale_factor,
        zoom_factors=zoom_factors,
    )

channel_list = print_channel_list(input_file, file_meta=file_meta)
stain_editor = edit_stain_dict(channel_list, name_setup=name_setup, use_saved=use_setup)

**Cell 5** (Read the stain table and run the batch)

Reads the confirmed stain table. If it isn't confirmed yet, it stops with a message: click **Confirm** in Cell 4, then run again from here.

In **batch mode** it then processes every image in the folder with the settings above (no viewers or plots) and saves each image's outputs to `<folder>_output/<image name>/`. The image-processing parameters for the batch are set in this cell. When the batch finishes, execution stops: the cells below are for single-file mode only.

In [ ]:
stain_dict, cyto_markers = stain_editor.result()
print(f"stain_dict = {stain_dict}")
print(f"cyto_markers = {cyto_markers}")

# --- Batch mode only (input_file is a folder) --------------------------------
# Image processing (set here because the tuning cells don't run in batch mode).
batch_sigma = 0.5                # smoothing strength: 0 = none, higher = smoother (noisy data)
batch_num_plateaus = 2           # contrast boost: raise to 3-4 for dim, low-contrast channels
batch_plateau_factor = 0.7       # contrast boost limit (0.5-1.0): lower = gentler, less noise amplified
batch_threshold_method = 'otsu'  # marker channels: 'otsu', 'median' (sparse) or 'huang' (dim)
batch_nuclei_min_snr = 3.0       # nuclei: positive from this many background-noise SDs
batch_cell_min_snr = 2.0         # cell body: positive from this many background-noise SDs
batch_correct_uneven_background = True

# Optional (slow) 3D exports, per image.
batch_export_vtk = True  # 3D volumes for ParaView
batch_export_stl = True  # 3D surface meshes per marker
batch_export_fea = True  # finite-element mesh (Abaqus .inp)

if batch_mode:
    batch_summary = run_batch_folder(
        input_file,
        roi_coords=ROI,
        stain_dict=stain_dict,
        cyto_markers=cyto_markers,
        name_setup=name_setup,
        use_setup=use_setup,
        automatic_contrast=automatic_contrast,
        interactive_roi=interactive_roi,
        nuclei_diameter=nuclei_diameter,
        cell_diameter=cell_diameter,
        scale_factor=scale_factor,
        zoom_factors=zoom_factors,
        trig_cellpose=trig_cellpose,
        trig_stardist=trig_stardist,
        trig_cellpose_cyto=trig_cellpose_cyto,
        multilabel=multilabel,
        aggregate_grow_factor=aggregate_grow_factor,
        size_reference=size_reference,
        iterative_split=iterative_split,
        oversize_factor=oversize_factor,
        max_split_iterations=max_split_iterations,
        merge_undersized=merge_undersized,
        undersize_factor=undersize_factor,
        sigma=batch_sigma,
        num_plateaus=batch_num_plateaus,
        plateau_factor=batch_plateau_factor,
        threshold_method=batch_threshold_method,
        cell_body_threshold=True,
        nuclei_min_snr=batch_nuclei_min_snr,
        cell_min_snr=batch_cell_min_snr,
        correct_uneven_background=batch_correct_uneven_background,
        export_vtk=batch_export_vtk,
        export_stl=batch_export_stl,
        export_fea=batch_export_fea,
        settings=settings,
        napari_module=napari,
        progress=tqdm,
    )
    display(batch_summary)
    # Stop 'Run All' here: the remaining cells are for single-file mode.
    raise BatchCompleted("Batch processing finished -- the remaining cells are for single-file mode only.")
else:
    print("Single-file mode -- continue with the next cells.")

# 3. Preparation and Preview

Prepare the image stack and set the display contrast/gamma of each channel.

**Cell 6** (Prepare the image stack)

Converts the image, builds the stain table (`stain_df`) and opens a napari viewer with each channel. Channels not used in the stain table are dropped.

In [ ]:
(im_final_stack, nuclei_radius, cell_radius, nuclei_volume, cell_volume,
 stain_df, viewer_0) = prepare_and_preview(
    img,
    nuclei_diameter, cell_diameter,
    stain_dict, file_meta,
    napari_module=napari, r_xyz=(r_X, r_Y, r_Z), progress=tqdm,
)

**Cell 7** (Contrast and gamma)

Reuses the saved `<name_setup>_setup.csv` if it exists (`use_setup = True`). Otherwise a napari window opens to adjust contrast/gamma per channel — or, with `automatic_contrast = True`, the contrast limits are picked automatically from each channel's histogram (gamma stays 1.0).

In [ ]:
stain_df, stain_complete_df, original_stain_complete_df = prepare_stain_settings(
    im_final_stack['Original image'],
    stain_df=stain_df,
    name_setup=name_setup,
    use_setup=use_setup,
    automatic_contrast=automatic_contrast,
    settings=settings,
    napari_module=napari,
    r_xyz=(r_X, r_Y, r_Z),
    progress=tqdm,
)

**Cell 8** (Review the stain settings)

Shows the complete stain settings table. Check the contrast limits and gamma before continuing.

In [ ]:
stain_complete_df

# 4. Image Processing and Segmentation

Run the cells in order: each step feeds the next and shows a histogram so you can follow its effect. Only Cells 13-15 have parameters to tune.

**Cell 9** (Normalize)

Scales each channel to 0-255.

In [ ]:
im_final_stack = run_normalize(
    im_final_stack,
    stain_complete_df=stain_complete_df,
)

**Cell 10** (Resample)

Resamples the image to the same voxel size in X, Y and Z.

In [ ]:
im_final_stack, r_zX, r_zY, r_zZ = run_resample(
    im_final_stack,
    stain_complete_df=stain_complete_df,
    zoom_factors=zoom_factors,
    meta=meta,
)

**Cell 11** (Denoise)

Median filter to remove noise.

In [ ]:
im_final_stack = run_denoise(
    im_final_stack,
    stain_complete_df=stain_complete_df,
)

**Cell 12** (Apply contrast and gamma)

Applies the contrast/gamma of each channel set in Cell 7.

In [ ]:
im_final_stack = run_contrast_gamma(
    im_final_stack,
    stain_complete_df=stain_complete_df,
)

**Cell 13** (Smoothing)

Gaussian smoothing. Raise `sigma` for noisy data; set it to 0 to skip smoothing.

In [ ]:
sigma = 0.5  # smoothing strength: 0 = none, 0.5 = mild, 1-2 = strong (noisy data)

im_final_stack = run_smooth(
    im_final_stack,
    stain_complete_df=stain_complete_df,
    sigma=sigma,
)

**Cell 14** (Contrast enhancement)

Histogram equalization to boost contrast before thresholding. The defaults suit most data.

In [ ]:
num_plateaus = 2      # contrast boost: raise to 3-4 for dim, low-contrast channels
plateau_factor = 0.7  # contrast boost limit (0.5-1.0): lower = gentler, less noise amplified

im_final_stack = run_equalize(
    im_final_stack,
    stain_complete_df=stain_complete_df,
    num_plateaus=num_plateaus,
    plateau_factor=plateau_factor,
)

**Cell 15** (Thresholding)

Separates signal from background in two parts:

1. **Marker channels** (`run_threshold`): a global threshold combined with a local one and a background-statistics one. `threshold_method` picks the global part. The histogram marks, per channel, the global (dashed orange) and final (solid red) thresholds. These masks decide which cells are positive for each marker.
2. **Nuclei and cell body** (`run_cell_body_threshold`): the NUCLEI channel and the cell body — the CYTOPLASM channel and/or the markers ticked as intracellular, combined — are thresholded against the **background noise** instead. Each channel is smoothed, its slowly varying background (uneven illumination, dimming with depth) is removed, and every voxel is measured in background-noise SDs. A voxel is positive from `nuclei_min_snr` / `cell_min_snr` SDs, as long as it is connected to clearly bright signal (or, for the cell body, to a nucleus). Even dim cytoplasm is therefore kept as cell body, while isolated noise is dropped. The thresholding runs on the image before the contrast/gamma of Cell 12, so dim signal cut by the display contrast still counts.

The resulting **nuclei mask** and **cell body mask** (whole cell, nucleus included) are what Cells 17 and 18 segment. The plot shows, for each, the intensity-above-background histogram with the thresholds, and a max projection with the mask outline.

- Dim cytoplasm missing → lower `cell_min_snr` (1-1.5). Background counted as cell → raise it (3-4).
- Dim nuclei missing → lower `nuclei_min_snr` (2). Nuclei merging through a halo → raise it (4-5).

In [ ]:
threshold_method = 'otsu'  # marker channels: 'otsu' (clear signal/background), 'median' (sparse) or 'huang' (dim/diffuse)

# Nuclei and cell body: positive from this many background-noise SDs. Lower = dimmer signal kept.
nuclei_min_snr = 3.0  # 2 = dim nuclei, 4-5 = nuclei merging through a halo
cell_min_snr = 2.0    # 1-1.5 = very dim cytoplasm, 3-4 = background counted as cell
correct_uneven_background = True  # remove slow background changes (uneven illumination, depth)

im_final_stack = run_threshold(
    im_final_stack,
    stain_complete_df=stain_complete_df,
    nuclei_diameter=nuclei_diameter,
    cell_diameter=cell_diameter,
    r_zX=r_zX, r_zY=r_zY, r_zZ=r_zZ,
    threshold_method=threshold_method,
    progress=tqdm,
)

im_final_stack = run_cell_body_threshold(
    im_final_stack,
    stain_complete_df=stain_complete_df,
    cyto_markers=cyto_markers,
    nuclei_diameter=nuclei_diameter,
    cell_diameter=cell_diameter,
    r_zX=r_zX, r_zY=r_zY, r_zZ=r_zZ,
    nuclei_min_snr=nuclei_min_snr,
    cell_min_snr=cell_min_snr,
    correct_uneven_background=correct_uneven_background,
    progress=tqdm,
)

**Cell 16** (Export histograms)

Saves the histograms of every processing step, plus a Parameters sheet, to `<file>_histograms.xlsx`.

In [ ]:
output_path = export_channel_histograms(
    im_final_stack, stain_complete_df, input_file,
    ROI_print=ROI_print, lazy_loading_used=used_lazy_loading,
    name_setup=name_setup,
    nuclei_diameter=nuclei_diameter, cell_diameter=cell_diameter,
    cyto_factor=cyto_factor, PCM_factor=PCM_factor,
    zoom_factors=zoom_factors, scale_factor=scale_factor,
    trig_cellpose=trig_cellpose, trig_stardist=trig_stardist,
    trig_cellpose_cyto=trig_cellpose_cyto,
    multilabel=multilabel, cyto_markers=cyto_markers,
    sigma=sigma, num_plateaus=num_plateaus, plateau_factor=plateau_factor,
    threshold_method=threshold_method,
    nuclei_min_snr=nuclei_min_snr, cell_min_snr=cell_min_snr,
    correct_uneven_background=correct_uneven_background,
    aggregate_grow_factor=aggregate_grow_factor,
    progress=tqdm,
)

**Cell 17** (Segment nuclei)

Segments the nuclei with the method chosen in Cell 3 (watershed, Cellpose or StarDist). The watershed works on the **nuclei mask** of Cell 15 and splits touching nuclei automatically, also where the nuclear signal dips between them, and always splits a nucleus whose slices show separate islands above and below its centre.

The nuclei are then refined by size (Cell 3, section 6):
1. a reference nucleus size is set from `nuclei_diameter` and/or the median detected size;
2. fragments smaller than `undersize_factor` × reference are merged into a touching neighbour — only if the result is still one plausible nucleus and there is no dark seam between them;
3. nuclei bigger than `oversize_factor` × reference are split again, with stronger settings each round ('aggressive' → 'very aggressive' → 'extreme').

Nuclei that can't be fixed are kept, never deleted. Each step and the final nuclei count are printed.

In [ ]:
im_segmentation_stack = segment_nuclei(
    im_final_stack,
    stain_df=stain_df,
    stain_complete_df=stain_complete_df,
    r_zxyz=(r_zX, r_zY, r_zZ),
    nuclei_diameter=nuclei_diameter,
    trig_stardist=trig_stardist,
    trig_cellpose=trig_cellpose,
    size_reference=size_reference,
    iterative_split=iterative_split,
    oversize_factor=oversize_factor,
    max_split_iterations=max_split_iterations,
    merge_undersized=merge_undersized,
    undersize_factor=undersize_factor,
    progress=tqdm,
)

**Cell 18** (Segment cytoplasm)

Splits the **cell body mask** of Cell 15 (CYTOPLASM channel and/or intracellular markers) into one cell per nucleus — or, with `trig_cellpose_cyto = True`, shapes the cells with Cellpose from the CYTOPLASM channel. Without a CYTOPLASM channel or intracellular markers, the cells are grown from the nuclei. Touching cells are split where the marker signal fades.

In [ ]:
im_segmentation_stack, stain_complete_df = segment_cytoplasm(
    im_final_stack,
    im_segmentation_stack=im_segmentation_stack,
    stain_df=stain_df,
    stain_complete_df=stain_complete_df,
    cyto_markers=cyto_markers,
    cyto_factor=cyto_factor,
    nuclei_diameter=nuclei_diameter,
    cell_diameter=cell_diameter,
    trig_cellpose_cyto=trig_cellpose_cyto,
    r_zxyz=(r_zX, r_zY, r_zZ),
    progress=tqdm,
)

**Cell 19** (Segment PCM)

The pericellular matrix: a shell around each cell.

In [ ]:
im_segmentation_stack = segment_pcm(
    im_segmentation_stack=im_segmentation_stack,
    stain_df=stain_df,
    cyto_markers=cyto_markers,
    cyto_factor=cyto_factor,
    PCM_factor=PCM_factor,
)

**Cell 20** (Assign cells to markers)

Links each cell to the marker channels where it is positive.

In [ ]:
im_segmentation_stack = assign_channel_labels(
    im_final_stack,
    im_segmentation_stack=im_segmentation_stack,
    stain_df=stain_df,
    progress=tqdm,
)

**Cell 21** (Detect aggregates)

Finds cell aggregates by growing each cell (`aggregate_grow_factor`) and checking which ones overlap.

In [ ]:
im_segmentation_stack = detect_aggregates(
    im_segmentation_stack,
    stain_df=stain_df,
    aggregate_grow_factor=aggregate_grow_factor,
)

**Cell 22** (View the results)

Opens napari viewers with every processing step and the segmentation. The "Thresh islands" layers show the connected regions of each threshold mask — for NUCLEI and CYTOPLASM these are the nuclei and cell body masks of Cell 15 (without a CYTOPLASM channel, the cell body mask has its own "Thresh islands CELL BODY" layer) — to check whether thresholding misses or breaks up nuclei and cells.

In [ ]:
viewer_0, viewer_1 = view_processing_results(
    im_final_stack,
    im_segmentation_stack=im_segmentation_stack,
    stain_df=stain_df,
    stain_complete_df=stain_complete_df,
    r_xyz=(r_X, r_Y, r_Z),
    r_zxyz=(r_zX, r_zY, r_zZ),
    napari_module=napari,
    progress=tqdm,
)

# 5. Quantification and Analysis

Measures every segmented cell and summarises the populations. Marker intensity is reported two ways: **positive cells** (only cells that passed that marker's threshold) and **all cells** (every segmented cell). Comparing the two shows how much the threshold shapes the average.

**Cell 23** (Measure the cells)

Mean/std intensity of every marker in each cell (nucleus + cytoplasm + PCM, all cells), then marker overlap, intensity, volume and position of each object (`labels_df`).

In [ ]:
percell_mean_df, percell_std_df = compute_percell_marker_intensity_df(
    im_segmentation_stack, im_final_stack,
    stain_complete_df=stain_complete_df,
    progress=tqdm,
)

labels_df = build_labels_df(
    im_segmentation_stack, im_final_stack,
    stain_complete_df=stain_complete_df,
    stain_df=stain_df,
    r_zX=r_zX, r_zY=r_zY, r_zZ=r_zZ,
    zooms=zoom_factors,
    multilabel=multilabel,
    percell_mean_df=percell_mean_df,
    percell_std_df=percell_std_df,
    progress=tqdm,
)

**Cell 24** (Population summary)

Counts and percentages per condition, with the positive-cells and all-cells intensity of each marker.

In [ ]:
print_population_summary(
    labels_df,
    stain_complete_df=stain_complete_df,
    stain_df=stain_df,
    progress=tqdm,
)

**Cell 25** (Full table for export)

The same measurements at the original (not resampled) resolution, used by the VTK and Excel exports.

In [ ]:
labels_full_df = build_full_labels_df(
    im_segmentation_stack, im_final_stack,
    stain_complete_df=stain_complete_df,
    r_zX=r_zX, r_zY=r_zY, r_zZ=r_zZ,
    zooms=zoom_factors,
    percell_mean_df=percell_mean_df,
    percell_std_df=percell_std_df,
    progress=tqdm,
)

**Cell 26** (Per-nucleus report)

Intensity distribution (KDE) of each marker inside every positive nucleus, plotted and saved as a PDF report.

In [ ]:
hist_data, intensity_ranges, fig, axes, x_grid = build_histogram_report(
    im_segmentation_stack, im_final_stack,
    stain_df=stain_df,
    stain_complete_df=stain_complete_df,
    input_file=input_file,
    progress=tqdm,
)

**Cell 27** (Spatial distribution)

2D and 3D plots of where the cells are.

In [ ]:
plot_spatial_distributions(
    labels_df,
    stain_complete_df=stain_complete_df,
    stain_df=stain_df,
    im_final_stack=im_final_stack,
    r_X=r_X, r_Y=r_Y, r_Z=r_Z,
    zoom_factors=zoom_factors,
    progress=tqdm,
)

**Cell 28** (Size distribution)

Volume and diameter of each population.

In [ ]:
plot_size_distributions(
    labels_df,
    stain_complete_df=stain_complete_df,
    stain_df=stain_df,
    progress=tqdm,
)

**Cell 29** (Marker intensity cloud)

Flow-cytometry-style plot: each cell's mean marker intensity (all cells) against its cytoplasm size, all markers together, with histograms on the sides. Saved as `<file>_marker_intensity.png`.

In [ ]:
plot_marker_intensity_clouds(
    percell_mean_df,
    stain_complete_df=stain_complete_df,
    im_segmentation_stack=im_segmentation_stack,
    r_xyz=(r_zX, r_zY, r_zZ),
    stain_df=stain_df,
    progress=tqdm,
    input_file=input_file,
)

# 6. Export

Writes the results to disk.

**Cell 30** (VTK volumes)

Labelled 3D volumes of nuclei, cytoplasm and PCM for ParaView. Each marker's all-cells intensity is included, so the meshes can be coloured by it.

In [ ]:
build_vtk_volumes(
    im_segmentation_stack,
    labels_full_df=labels_full_df,
    stain_complete_df=stain_complete_df,
    input_file=input_file,
    r_xyz=(r_X, r_Y, r_Z),
    zoom_factors=zoom_factors,
    progress=tqdm,
)

**Cell 31** (STL meshes)

One 3D surface mesh per marker.

In [ ]:
export_marker_stl(
    im_segmentation_stack,
    stain_df=stain_df,
    stain_complete_df=stain_complete_df,
    input_file=input_file,
    r_xyz=(r_X, r_Y, r_Z),
    zoom_factors=zoom_factors,
    progress=tqdm,
)

**Cell 32** (Single-cell export, optional)

Set `nuc_3D_export = True` to export one cell (nucleus, cytoplasm, PCM and marker channels) as a VTK sub-volume for ParaView. When the cell runs, a text box asks for the cell's label number.

In [ ]:
nuc_3D_export = False  # True = export one cell (you'll be asked for its label number)

if nuc_3D_export:
    nuc_label = int(input("Enter nucleus label number to export: "))
    export_nucleus_vtk_crop(
        nuc_label=nuc_label,
        im_segmentation_stack=im_segmentation_stack,
        im_final_stack=im_final_stack,
        stain_df=stain_df,
        input_file=input_file,
        r_zxyz=(r_zX, r_zY, r_zZ),
        zoom_factors=zoom_factors,
    )

**Cell 33** (Excel export)

The full measurements table, stain settings and parameters in `<file>_segmentation.xlsx`, with both positive-cells and all-cells statistics.

In [ ]:
output_path = export_quantification_to_excel(
    input_file,
    original_stain_complete_df,
    labels_full_df,
    progress=tqdm,
)

**Cell 34** (FEA mesh)

Abaqus `.inp` mesh of the segmentation for finite element analysis.

In [ ]:
export_fea_mesh(
    im_segmentation_stack,
    input_file=input_file,
    progress=tqdm,
)